In [ ]:
import json
import re
from pathlib import Path


NUM = r"(?:\d+\s+\d+/\d+|\d+/\d+|\d+(?:\.\d+)?)"
RANGE = rf"(?P<a>{NUM})(?:\s*[~\-–]\s*(?P<b>{NUM}))?"

UNIT_PATTERN = (
    r"큰술|작은술|티스푼|스푼|Ts|TS|ts|T|t|"
    r"kg|KG|g|G|ml|mL|ML|L|l|"
    r"종이컵|소주잔|컵|개|마리|대|쪽|줌|꼬집|팩|봉지|장|톨|"
    r"뿌리|통|인분|움큼|주먹|모|알|스틱|캔|병|포기"
)

MEASURE_RE = re.compile(
    rf"(?<!\w){RANGE}\s*(?P<unit>{UNIT_PATTERN})(?!\w)"
)

QUAL_RE = re.compile(
    r"\s*(약간|조금|적당량|듬뿍|톡톡|취향껏|적당히)\s*$"
)

# 단위 통일
UNIT_ALIASES = {
    "T": "큰술",
    "Ts": "큰술",
    "TS": "큰술",
    "스푼": "큰술",

    "t": "작은술",
    "ts": "작은술",
    "티스푼": "작은술",

    "mL": "ml",
    "ML": "ml",

    "G": "g",
    "KG": "kg",

    "l": "L",
}


# 재료명 통일
ALIASES = {
    "달걀": "계란",
    "고추가루": "고춧가루",

    "후추가루": "후추",
    "후춧가루": "후추",

    "소세지": "소시지",

    "굴 소스": "굴소스",
    "토마토 소스": "토마토소스",

    "올리브 오일": "올리브유",

    "깨소금": "깨",
}


SEASONING_NAMES = {
    "간장",
    "진간장",
    "국간장",
    "양조간장",
    "맛간장",

    "고추장",
    "된장",
    "쌈장",

    "굴소스",
    "돈까스소스",
    "토마토소스",
    "케첩",
    "마요네즈",

    "고춧가루",

    "설탕",
    "황설탕",
    "소금",
    "후추",
    "식초",

    "맛술",
    "미림",
    "청주",

    "요리당",
    "물엿",
    "올리고당",
    "매실액",
    "꿀",

    "참기름",
    "들기름",
    "식용유",
    "올리브유",
    "고추기름",

    "액젓",
    "멸치액젓",
    "까나리액젓",

    "참깨",
    "깨",
}

#숫자 , 단위 처리
def to_number(text):
    if text is None:
        return None

    text = text.strip()

    # 1 1/2
    if " " in text and "/" in text:
        whole, fraction = text.split(None, 1)
        numerator, denominator = fraction.split("/")

        return (
            float(whole)
            + float(numerator) / float(denominator)
        )

    # 1/2
    if "/" in text:
        numerator, denominator = text.split("/")

        return (
            float(numerator)
            / float(denominator)
        )

    return float(text)


def measure_to_dict(match):
    amount_min = to_number(
        match.group("a")
    )

    if match.group("b"):
        amount_max = to_number(
            match.group("b")
        )
    else:
        amount_max = amount_min

    unit_raw = match.group("unit")

    unit_normalized = UNIT_ALIASES.get(
        unit_raw,
        unit_raw
    )

    return {
        "amount_text": match.group(0).strip(),
        "amount_min": amount_min,
        "amount_max": amount_max,
        "unit_raw": unit_raw,
        "unit_normalized": unit_normalized,
    }

#재료 이름 정규화
def normalize_name_and_detail(name):

    name = (
        name
        or ""
    ).strip()

    name = re.sub(
        r"\s+",
        " ",
        name
    )

    name = name.replace(
        "혹은",
        "또는"
    )

    preparation = []
    detail_parts = []


    # 다진마늘 → 마늘
    preparation_rules = [
        (
            r"^다진\s*(마늘|양파|파|대파|생강)$",
            "다진"
        ),
        (
            r"^간\s*(마늘|양파|생강)$",
            "간"
        ),
        (
            r"^통\s*(마늘)$",
            "통"
        ),
        (
            r"^불린\s*(찹쌀|쌀|미역|당면)$",
            "불린"
        ),
        (
            r"^마른\s+(.+)$",
            "마른"
        ),
        (
            r"^고운\s*(고춧가루|고추가루)$",
            "고운"
        ),
    ]

    for pattern, prep in preparation_rules:

        match = re.match(
            pattern,
            name
        )

        if match:
            name = match.group(1)

            preparation.append(
                prep
            )

            break


    # 생닭 → 닭
    if name == "생닭":
        name = "닭"

        preparation.append(
            "생"
        )


    # 돼지고기 찌개용 → 돼지고기
    descriptor_patterns = [
        r"^(돼지고기)\s+(.+)$",
        r"^(소고기)\s+(.+)$",
        r"^(오징어)\s+(큰\s*사이즈|대|중|소)$",
        r"^(모짜렐라치즈)\s+(듬뿍|적당량|약간)$",
    ]

    for pattern in descriptor_patterns:

        match = re.match(
            pattern,
            name
        )

        if match:

            name = match.group(1)

            detail_parts.append(
                match.group(2).strip()
            )

            break


    # 국거리용 / 찌개용 등
    match = re.match(
        r"^(.+?)\s+"
        r"(국거리용|"
        r"찌개용(?:\s+또는\s+.+)?|"
        r"큰\s*사이즈|"
        r"듬뿍)$",
        name
    )

    if match:

        name = match.group(1).strip()

        detail_parts.append(
            match.group(2).strip()
        )


    # alias 통일
    name = ALIASES.get(
        name,
        name
    )

    name = re.sub(
        r"\s+",
        " ",
        name
    ).strip()


    detail = "; ".join(
        detail_parts
    )

    return (
        name,
        detail,
        preparation
    )


#일반 재료 , 양념 분류
def classify_item(
    name,
    group="",
    original_type="ingredient"
):

    group_text = (
        group
        or ""
    ).replace(
        " ",
        ""
    )

    # 원래 seasonings에 있으면 양념
    if original_type == "seasoning":
        return "seasoning"

    # 그룹명 기준
    if any(
        keyword in group_text
        for keyword in [
            "양념",
            "소스",
            "드레싱"
        ]
    ):
        return "seasoning"

    # 이름 기준
    if name in SEASONING_NAMES:
        return "seasoning"

    return "ingredient"

def parse_food_item(
    raw,
    group="",
    original_type="ingredient"
):

    raw = (
        raw
        or ""
    ).replace(
        "구매",
        ""
    ).strip()

    raw = re.sub(
        r"\s+",
        " ",
        raw
    )


    measurements = list(
        MEASURE_RE.finditer(
            raw
        )
    )

    qualitative_amount = None


    # 수량 + 단위가 있는 경우
    if measurements:

        primary_match = measurements[-1]

        primary = measure_to_dict(
            primary_match
        )

        secondary_amounts = [
            measure_to_dict(match)
            for match in measurements[:-1]
        ]

        name_text = raw[
            :measurements[0].start()
        ].strip(
            " ,/"
        )

        if len(measurements) > 1:

            middle_text = raw[
                measurements[0].end()
                :
                primary_match.start()
            ].strip(
                " ,/"
            )

        else:
            middle_text = ""

        trailing_text = raw[
            primary_match.end():
        ].strip(
            " ,/"
        )

        detail_extra = " ".join(
            x
            for x in [
                middle_text,
                trailing_text
            ]
            if x
        )


    # 약간 / 조금 / 적당량
    else:

        qual_match = QUAL_RE.search(
            raw
        )

        if qual_match:

            qualitative_amount = (
                qual_match.group(1)
            )

            name_text = raw[
                :qual_match.start()
            ].strip(
                " ,/"
            )

        else:
            name_text = raw

        primary = {
            "amount_text":
                qualitative_amount or "",

            "amount_min":
                None,

            "amount_max":
                None,

            "unit_raw":
                None,

            "unit_normalized":
                None,
        }

        secondary_amounts = []
        detail_extra = ""


    (
        name_normalized,
        detail,
        preparation

    ) = normalize_name_and_detail(
        name_text
    )


    if detail_extra:

        detail = "; ".join(
            x
            for x in [
                detail,
                detail_extra
            ]
            if x
        )


    item_type = classify_item(
        name_normalized,
        group,
        original_type
    )


    return {
        "name_raw":
            name_text,

        "name_normalized":
            name_normalized,

        "detail":
            detail,

        "preparation":
            preparation,

        "group":
            group,

        "item_type":
            item_type,

        "amount_text":
            primary["amount_text"],

        "amount_min":
            primary["amount_min"],

        "amount_max":
            primary["amount_max"],

        "unit_raw":
            primary["unit_raw"],

        "unit_normalized":
            primary["unit_normalized"],

        "qualitative_amount":
            qualitative_amount,

        "secondary_amounts":
            secondary_amounts,

        "has_alternative":
            (
                "또는" in name_text
                or "혹은" in name_text
            ),

        "raw":
            raw,
    }

#중복 제거
def dedupe_items(items):

    seen = set()
    result = []

    for item in items:

        key = (
            item["name_normalized"],
            item["amount_text"],
            item["group"],
            item["raw"],
        )

        if key in seen:
            continue

        seen.add(
            key
        )

        result.append(
            item
        )

    return result

def preprocess_recipe(recipe):

    ingredients_clean = []
    seasonings_clean = []


    fields = [
        (
            "ingredient",
            "ingredients"
        ),
        (
            "seasoning",
            "seasonings"
        ),
    ]


    for original_type, field in fields:

        for item in (
            recipe.get(field)
            or []
        ):

            if isinstance(
                item,
                dict
            ):

                raw = (
                    item.get("raw")
                    or
                    " ".join(
                        str(x)
                        for x in [
                            item.get("name"),
                            item.get("amount")
                        ]
                        if x
                    )
                )

                group = (
                    item.get("group")
                    or ""
                )

            else:
                raw = str(item)
                group = ""


            parsed = parse_food_item(
                raw,
                group,
                original_type
            )

            if not parsed[
                "name_normalized"
            ]:
                continue


            if (
                parsed["item_type"]
                == "seasoning"
            ):

                seasonings_clean.append(
                    parsed
                )

            else:

                ingredients_clean.append(
                    parsed
                )


    ingredients_clean = dedupe_items(
        ingredients_clean
    )

    seasonings_clean = dedupe_items(
        seasonings_clean
    )


    result = dict(
        recipe
    )

    result["recipe_uid"] = (
        f"{recipe.get('source', 'unknown')}_"
        f"{recipe.get('source_id', '')}"
    )


    # 정제 결과 추가
    result["ingredients_clean"] = (
        ingredients_clean
    )

    result["seasonings_clean"] = (
        seasonings_clean
    )


    return result

INPUT_DIR = Path(
    r"C:\Users\Playdata\Desktop\mle-01-p2-team2\홍기표\input"
)

OUTPUT_DIR = Path(
    r"C:\Users\Playdata\Desktop\mle-01-p2-team2\홍기표\output"
)


INPUT_FILE = (
    INPUT_DIR
    / "recipes_10000.jsonl"
)

OUTPUT_FILE = (
    OUTPUT_DIR
    / "recipes_10000_cleaned.jsonl"
)


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


with INPUT_FILE.open(
    "r",
    encoding="utf-8"
) as src, OUTPUT_FILE.open(
    "w",
    encoding="utf-8"
) as out:

    for line_number, line in enumerate(
        src,
        start=1
    ):

        try:

            recipe = json.loads(
                line
            )

            cleaned = preprocess_recipe(
                recipe
            )

            out.write(
                json.dumps(
                    cleaned,
                    ensure_ascii=False
                )
                + "\n"
            )

        except Exception as e:

            print(
                f"{line_number}번째 줄 오류: {e}"
            )
